In [0]:
%pip install dbt-databricks
dbutils.library.restartPython()

In [0]:
%sh
cat > /Workspace/Users/alejandrowsky.dev@outlook.com/delitos_cdmx_DE_to_ML/dbt/delitos_cdmx/profiles.yml << 'EOF'
delitos_cdmx:
  target: dev
  outputs:
    dev:
      type: databricks
      catalog: delitos
      schema: gold
      host: ""
      http_path: ""
      token: ""
      threads: 4
EOF

In [0]:
%sh
cat /Workspace/Users/alejandrowsky.dev@outlook.com/delitos_cdmx_DE_to_ML/dbt/delitos_cdmx/profiles.yml

In [0]:
%sql
SHOW TABLES IN delitos.silver

In [0]:
%sql
SHOW TABLES IN delitos.gold

In [0]:
%sh
find /Workspace/Users/alejandrowsky.dev@outlook.com/delitos_cdmx_DE_to_ML/dbt/ -name "*.sql" 2>/dev/null

In [0]:
%sh
cat > /Workspace/Users/alejandrowsky.dev@outlook.com/delitos_cdmx_DE_to_ML/dbt/delitos_cdmx/profiles/silver_base.sql << 'EOF'
{{ config(
    materialized='view',
    catalog='delitos',
    schema='gold'
) }}

SELECT *
FROM {{ source('silver', 'delitos_cdmx_silver') }}
EOF

In [0]:
%sh
cat > /Workspace/Users/alejandrowsky.dev@outlook.com/delitos_cdmx_DE_to_ML/dbt/delitos_cdmx/profiles/models/sources.yml << 'EOF'
version: 2

sources:
  - name: silver
    catalog: delitos
    schema: silver
    tables:
      - name: delitos_cdmx_silver
EOF

In [0]:
%sh
cd /Workspace/Users/alejandrowsky.dev@outlook.com/delitos_cdmx_DE_to_ML/dbt/delitos_cdmx/profiles/delitos_cdmx && dbt run --select silver_base --profiles-dir /Workspace/Users/alejandrowsky.dev@outlook.com/delitos_cdmx_DE_to_ML/dbt/delitos_cdmx/profiles/delitos_cdmx

In [0]:
%sh
cat > /Workspace/Users/alejandrowsky.dev@outlook.com/delitos_cdmx_DE_to_ML/dbt/delitos_cdmx/profiles/models/gold_features.sql << 'EOF'
{{ config(
    materialized='table',
    catalog='delitos',
    schema='gold'
) }}

WITH base AS (
    SELECT *
    FROM delitos.silver.delitos_cdmx
),

conteos AS (
    SELECT
        AlcaldiaHechos,
        hora_del_dia,
        dia_semana,
        mes_hechos_num,
        COUNT(*) OVER (PARTITION BY AlcaldiaHechos, hora_del_dia) AS conteo_alcaldia_hora,
        COUNT(*) OVER (PARTITION BY AlcaldiaHechos, dia_semana)   AS conteo_alcaldia_dia,
        COUNT(*) OVER (PARTITION BY AlcaldiaHechos, mes_hechos_num) AS conteo_alcaldia_mes
    FROM base
)

SELECT
    b.categoria_delito,
    b.hora_del_dia,
    b.dia_semana,
    b.dia_mes,
    b.trimestre,
    b.mes_hechos_num,
    b.AlcaldiaHechos,
    b.longitud,
    b.latitud,
    b.dias_para_registro,
    c.conteo_alcaldia_hora,
    c.conteo_alcaldia_dia,
    c.conteo_alcaldia_mes
FROM base b
JOIN conteos c
    ON  b.AlcaldiaHechos  = c.AlcaldiaHechos
    AND b.hora_del_dia    = c.hora_del_dia
    AND b.dia_semana      = c.dia_semana
    AND b.mes_hechos_num  = c.mes_hechos_num
EOF

In [0]:
%sh
cd /Workspace/Users/alejandrowsky.dev@outlook.com/delitos_cdmx_DE_to_ML/dbt/delitos_cdmx/profiles/delitos_cdmx && dbt run --select silver_base --profiles-dir /Workspace/Users/alejandrowsky.dev@outlook.com/delitos_cdmx_DE_to_ML/dbt/delitos_cdmx/profiles/delitos_cdmx